# Crop vs Weed Detection — CNN (VGG16 Transfer Learning)

End-to-end training notebook companion to *Efficient Crop vs Weed Detection in Precision Agriculture: A CNN Approach for Real-Time Decision Making* (INOCON 2024).

Run this notebook top-to-bottom on Google Colab (enable a GPU runtime: `Runtime > Change runtime type > GPU`) or locally.

In [ ]:
!git clone https://github.com/<your-username>/crop-weed-detection-cnn.git
%cd crop-weed-detection-cnn
!pip install -q -r requirements.txt

## 1. Get the dataset

Download the WeedCrop dataset (Roboflow / Kaggle, YOLOv5 PyTorch export) and place it at
`data/raw/WeedCrop.v1i.yolov5pytorch/`. If you have a `kaggle.json` API token you can
fetch it programmatically instead — see `data/README.md`.

In [ ]:
# Example: unzip a manually uploaded dataset archive
# !unzip -q /content/kaggle_dataset.zip -d data/raw/

## 2. Convert YOLO bounding boxes into a classification dataset

In [ ]:
!python src/data_preprocessing.py \
    --source data/raw/WeedCrop.v1i.yolov5pytorch \
    --output data/processed \
    --padding 0.08

## 3. Train the VGG16-based CNN

This reproduces the paper's architecture: a frozen VGG16 convolutional base
with a custom 2-layer fully-connected head (256, 128 units).

In [ ]:
!python src/train.py \
    --data data/processed \
    --epochs 25 \
    --batch-size 32 \
    --lr 1e-4

### Optional: fine-tune the later VGG16 blocks for a few more epochs
This can push accuracy closer to the paper's reported 94-95%.

In [ ]:
!python src/train.py \
    --data data/processed \
    --epochs 10 \
    --batch-size 32 \
    --lr 1e-5 \
    --fine-tune-at 15

## 4. Evaluate on the held-out test set

In [ ]:
!python src/evaluate.py --model models/best_model.keras --data data/processed --split test

## 5. View training curves and confusion matrix

In [ ]:
from IPython.display import Image, display
display(Image('results/accuracy_plot.png'))
display(Image('results/loss_plot.png'))
display(Image('results/confusion_matrix.png'))

## 6. Try it on a single image

In [ ]:
!python src/predict.py --model models/best_model.keras --image data/processed/test/weed/<some_image>.jpg